# Comparación de modelos de regresión

Compara Ridge, SVR (RBF) y XGBoost sobre las mismas features (VAE μ + T1w)
y el mismo split de datos. Cada modelo se optimiza con Optuna (5-fold CV sobre trainval)
para una comparación justa, y luego se evalúa en el test set retenido.

Usa los artefactos ya generados por el pipeline principal (embeddings, splits).

In [1]:
import sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=FutureWarning)

_cwd = Path.cwd()
ROOT = _cwd if (_cwd / "src").exists() else _cwd.parent
REPO = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import Paths, ExperimentConfig
from src.cohort import build_final_cohort_df
from src.splits import load_splits
from src.xgb_train import build_feats, clean_xy
from src.metrics import regression_metrics

paths = Paths(
    excel_path=REPO / "Data" / "datos-redlat.xlsx",
    fc_folder=REPO / "Data" / "fc_mats",
    t1w_csv_path=REPO / "Data" / "Redlat_VGM_AAL_.csv",
    out_dir=REPO / "Outputs",
)
cfg = ExperimentConfig(seed=42)
SEED = cfg.seed
print("Ready")

/home/nico/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ready


## 1. Carga de datos y artefactos

In [2]:
cohort = build_final_cohort_df(
    excel_path=paths.excel_path,
    fc_folder=paths.fc_folder,
    t1w_csv_path=paths.t1w_csv_path,
    diagnoses_to_use=cfg.diagnoses_to_use,
)

all_splits = load_splits(paths.out_dir / "splits" / f"splits_seed{SEED}_test{cfg.test_size}.json")
train_ids = set(all_splits["holdout"]["trainval_ids"])
test_ids  = set(all_splits["holdout"]["test_ids"])
folds     = all_splits["folds"]  # 5-fold CV indices over trainval

mask_tr = cohort["record_id"].isin(train_ids)
mask_te = cohort["record_id"].isin(test_ids)

t1_cols = [c for c in cohort.columns if c.startswith("t1_")]
T1_tr = cohort.loc[mask_tr, t1_cols].values.astype(np.float32)
T1_te = cohort.loc[mask_te, t1_cols].values.astype(np.float32)
y_tr  = cohort.loc[mask_tr, "age"].values.astype(float)
y_te  = cohort.loc[mask_te, "age"].values.astype(float)

# Record IDs in trainval (needed to map fold IDs to row indices)
ids_tr = cohort.loc[mask_tr, "record_id"].values

Z_tr = np.load(paths.out_dir / "embeddings" / "mu_trainval.npy")
Z_te = np.load(paths.out_dir / "embeddings" / "mu_test.npy")

X_tr = build_feats(Z=Z_tr, T1=T1_tr)
X_te = build_feats(Z=Z_te, T1=T1_te)
X_tr, y_tr = clean_xy(X_tr, y_tr)
X_te, y_te = clean_xy(X_te, y_te)

print(f"Trainval: {X_tr.shape}  |  Test: {X_te.shape}  |  Folds: {len(folds)}")

Trainval: (1120, 180)  |  Test: (125, 180)  |  Folds: 5


## 2. Funciones auxiliares de CV

In [3]:
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor


def _fold_indices(folds, ids_tr):
    """Convert fold ID lists to row-index arrays matching X_tr / y_tr."""
    id_to_idx = {rid: i for i, rid in enumerate(ids_tr)}
    out = []
    for f in folds:
        tr_idx = np.array([id_to_idx[rid] for rid in f["train_ids"] if rid in id_to_idx])
        va_idx = np.array([id_to_idx[rid] for rid in f["val_ids"]   if rid in id_to_idx])
        out.append((tr_idx, va_idx))
    return out


cv_idx = _fold_indices(folds, ids_tr)
print(f"CV folds ready: {[len(va) for _, va in cv_idx]} val sizes")


def cv_mae(model_fn):
    """Run 5-fold CV and return mean MAE. model_fn(X_train, y_train) -> fitted model."""
    maes = []
    for tr_i, va_i in cv_idx:
        scaler = StandardScaler()
        Xf_tr = scaler.fit_transform(X_tr[tr_i])
        Xf_va = scaler.transform(X_tr[va_i])
        model = model_fn(Xf_tr, y_tr[tr_i])
        pred = model.predict(Xf_va)
        maes.append(mean_absolute_error(y_tr[va_i], pred))
    return float(np.mean(maes))

CV folds ready: [224, 224, 224, 224, 224] val sizes


## 3. Optuna — SVR (RBF)

In [4]:
def svr_objective(trial):
    C       = trial.suggest_float("C", 0.1, 500.0, log=True)
    epsilon = trial.suggest_float("epsilon", 0.001, 2.0, log=True)
    gamma   = trial.suggest_categorical("gamma", ["scale", "auto"])
    
    def make_svr(X, y):
        m = SVR(kernel="rbf", C=C, epsilon=epsilon, gamma=gamma)
        m.fit(X, y)
        return m
    
    return cv_mae(make_svr)


sampler_svr = optuna.samplers.TPESampler(seed=SEED)
study_svr = optuna.create_study(direction="minimize", sampler=sampler_svr,
                                 study_name="svr_rbf")
study_svr.optimize(svr_objective, n_trials=100, show_progress_bar=True)

print(f"\nBest SVR CV MAE: {study_svr.best_value:.4f}")
print(f"Best params: {study_svr.best_params}")


  0%|          | 0/100 [00:00<?, ?it/s]


Best trial: 0. Best value: 6.59175:   0%|          | 0/100 [00:00<?, ?it/s]


Best trial: 0. Best value: 6.59175:   1%|          | 1/100 [00:00<00:22,  4.41it/s]


Best trial: 0. Best value: 6.59175:   1%|          | 1/100 [00:00<00:22,  4.41it/s]


Best trial: 0. Best value: 6.59175:   2%|▏         | 2/100 [00:00<00:23,  4.12it/s]

...


Best trial: 63. Best value: 6.21806:  99%|█████████▉| 99/100 [00:30<00:00,  3.06it/s]


Best trial: 63. Best value: 6.21806:  99%|█████████▉| 99/100 [00:30<00:00,  3.06it/s]


Best trial: 63. Best value: 6.21806: 100%|██████████| 100/100 [00:30<00:00,  3.16it/s]


Best trial: 63. Best value: 6.21806: 100%|██████████| 100/100 [00:30<00:00,  3.24it/s]


Best SVR CV MAE: 6.2181
Best params: {'C': 24.838197245440746, 'epsilon': 1.9434322439739649, 'gamma': 'auto'}


## 4. Optuna — Ridge

In [5]:
def ridge_objective(trial):
    alpha = trial.suggest_float("alpha", 0.001, 1000.0, log=True)
    
    def make_ridge(X, y):
        m = Ridge(alpha=alpha, random_state=SEED)
        m.fit(X, y)
        return m
    
    return cv_mae(make_ridge)


sampler_ridge = optuna.samplers.TPESampler(seed=SEED)
study_ridge = optuna.create_study(direction="minimize", sampler=sampler_ridge,
                                   study_name="ridge")
study_ridge.optimize(ridge_objective, n_trials=100, show_progress_bar=True)

print(f"\nBest Ridge CV MAE: {study_ridge.best_value:.4f}")
print(f"Best params: {study_ridge.best_params}")


  0%|          | 0/100 [00:00<?, ?it/s]


Best trial: 0. Best value: 6.97252:   0%|          | 0/100 [00:00<?, ?it/s]


Best trial: 1. Best value: 6.57788:   1%|          | 1/100 [00:00<00:02, 34.75it/s]


Best trial: 1. Best value: 6.57788:   2%|▏         | 2/100 [00:00<00:02, 46.72it/s]


Best trial: 1. Best value: 6.57788:   3%|▎         | 3/100 [00:00<00:01, 53.33it/s]

...


Best trial: 81. Best value: 6.53614:  97%|█████████▋| 97/100 [00:01<00:00, 67.98it/s]


Best trial: 81. Best value: 6.53614:  98%|█████████▊| 98/100 [00:01<00:00, 67.98it/s]


Best trial: 81. Best value: 6.53614:  99%|█████████▉| 99/100 [00:01<00:00, 67.98it/s]


Best trial: 81. Best value: 6.53614: 100%|██████████| 100/100 [00:01<00:00, 68.98it/s]


Best Ridge CV MAE: 6.5361
Best params: {'alpha': 267.7011431833793}


## 5. Evaluación final en test (todos optimizados)

In [6]:
# --- SVR optimizado ---
svr_params = study_svr.best_params
scaler_svr = StandardScaler().fit(X_tr)
svr_model = SVR(kernel="rbf", **svr_params)
svr_model.fit(scaler_svr.transform(X_tr), y_tr)
svr_pred = svr_model.predict(scaler_svr.transform(X_te))

# --- Ridge optimizado ---
ridge_params = study_ridge.best_params
scaler_ridge = StandardScaler().fit(X_tr)
ridge_model = Ridge(**ridge_params, random_state=SEED)
ridge_model.fit(scaler_ridge.transform(X_tr), y_tr)
ridge_pred = ridge_model.predict(scaler_ridge.transform(X_te))

# --- XGBoost (ya optimizado en pipeline principal) ---
with open(paths.out_dir / "optuna" / "xgb_best_cv.json") as f:
    xgb_best = json.load(f)
xgb_params = xgb_best["best_params"]
xgb_params["random_state"] = SEED
xgb_params["n_jobs"] = -1
xgb_model = XGBRegressor(**xgb_params)
xgb_model.fit(X_tr, y_tr, verbose=False)
xgb_pred = xgb_model.predict(X_te)

# --- Tabla de resultados ---
rows = []
for name, pred, cv_mae_val in [
    ("Ridge (opt.)",    ridge_pred, study_ridge.best_value),
    ("SVR RBF (opt.)",  svr_pred,   study_svr.best_value),
    ("XGBoost (opt.)",  xgb_pred,   xgb_best["best_value"]),
]:
    m = regression_metrics(y_te, pred)
    m["Model"]  = name
    m["CV_MAE"] = round(cv_mae_val, 4)
    rows.append(m)
    print(f"{name:20s} | CV={cv_mae_val:.2f}  MAE={m['MAE']:.2f}  RMSE={m['RMSE']:.2f}  R²={m['R2']:.3f}  r={m['Pearson']:.3f}")

df_final = pd.DataFrame(rows)[["Model", "CV_MAE", "MAE", "RMSE", "R2", "Pearson"]]
df_final

Ridge (opt.)         | CV=6.54  MAE=5.75  RMSE=7.25  R²=0.403  r=0.639
SVR RBF (opt.)       | CV=6.22  MAE=5.65  RMSE=7.02  R²=0.441  r=0.664
XGBoost (opt.)       | CV=6.36  MAE=5.62  RMSE=7.20  R²=0.411  r=0.644


,Model,CV_MAE,MAE,RMSE,R2,Pearson
0,Ridge (opt.),6.5361,5.753391,7.254336,0.402786,0.639293
1,SVR RBF (opt.),6.2181,5.648224,7.020701,0.440635,0.663806
2,XGBoost (opt.),6.3567,5.620665,7.204005,0.411044,0.644342


## 6. Guardar resultados y parámetros

In [7]:
out_dir = paths.out_dir / "experiments"
out_dir.mkdir(parents=True, exist_ok=True)

# Table
df_final.to_json(out_dir / "model_comparison.json", orient="records", indent=2)

# Best params for reproducibility
params_out = {
    "svr_best":   {"best_value": study_svr.best_value,   "best_params": study_svr.best_params},
    "ridge_best": {"best_value": study_ridge.best_value,  "best_params": study_ridge.best_params},
    "xgb_best":   {"best_value": xgb_best["best_value"],  "best_params": xgb_best["best_params"]},
}
with open(out_dir / "regressor_optuna_params.json", "w") as f:
    json.dump(params_out, f, indent=2)

print("Saved to", out_dir)

Saved to /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Outputs/experiments
